In [117]:
import pandas as pd
import numpy as np
import time

from pathlib import Path

from sklearn.model_selection import train_test_split

from sklearn.compose import ColumnTransformer
from sklearn.preprocessing import OneHotEncoder, StandardScaler

from sklearn.pipeline import Pipeline

from sklearn.linear_model import LogisticRegression
from sklearn.tree import DecisionTreeClassifier
from sklearn.neighbors import KNeighborsClassifier
from sklearn.ensemble import (
    RandomForestClassifier,
    GradientBoostingClassifier
)

from xgboost import XGBClassifier

from sklearn.metrics import (
    accuracy_score,
    precision_score,
    recall_score,
    f1_score,
    roc_auc_score,
    average_precision_score,
    confusion_matrix,
    classification_report
)

In [118]:
BASE_DIR = Path.cwd().parent

DATA_PATH = (
    BASE_DIR
    / "data"
    / "processed"
    / "feature_engineered_data.csv"
)

df = pd.read_csv(DATA_PATH)

print("Dataset shape:", df.shape)

df.head()

Dataset shape: (7043, 38)


,customerID,gender,SeniorCitizen,Partner,Dependents,tenure,PhoneService,MultipleLines,InternetService,OnlineSecurity,...,MonthlyChargeGroup,TotalChargeGroup,ChargePerTenure,HighValueCustomer,MonthToMonthContract,ElectronicCheck,NoTechSupport,NoOnlineSecurity,ServiceRiskIndicator,HighRiskCustomer
0,7590-VHVEG,Female,0,Yes,No,1,No,No phone service,DSL,No,...,Low,Low,29.850000,0,1,1,1,1,1,0
1,5575-GNVDE,Male,0,No,No,34,Yes,No,DSL,Yes,...,Medium,High,55.573529,0,0,0,1,0,0,0
2,3668-QPYBK,Male,0,No,No,2,Yes,No,DSL,Yes,...,Medium,Low,54.075000,0,1,0,1,0,0,0
3,7795-CFOCW,Male,0,No,No,45,No,No phone service,DSL,Yes,...,Medium,High,40.905556,0,0,0,0,0,0,0
4,9237-HQITU,Female,0,No,No,2,Yes,No,Fiber optic,No,...,High,Low,75.825000,0,1,1,1,1,1,0


In [119]:
df["Churn"]

0       0
1       0
2       1
3       0
4       1
       ..
7038    0
7039    0
7040    0
7041    1
7042    0
Name: Churn, Length: 7043, dtype: int64

In [120]:
print("Missing values:", df.isnull().sum().sum())

print("Duplicate rows:", df.duplicated().sum())

print("\nColumns:")
print(df.columns.tolist())

Missing values: 0
Duplicate rows: 0

Columns:
['customerID', 'gender', 'SeniorCitizen', 'Partner', 'Dependents', 'tenure', 'PhoneService', 'MultipleLines', 'InternetService', 'OnlineSecurity', 'OnlineBackup', 'DeviceProtection', 'TechSupport', 'StreamingTV', 'StreamingMovies', 'Contract', 'PaperlessBilling', 'PaymentMethod', 'MonthlyCharges', 'TotalCharges', 'Churn', 'TenureGroup', 'IsNewCustomer', 'IsLongTermCustomer', 'NumberOfServices', 'OptionalServiceCount', 'HasInternetService', 'IsFiberCustomer', 'MonthlyChargeGroup', 'TotalChargeGroup', 'ChargePerTenure', 'HighValueCustomer', 'MonthToMonthContract', 'ElectronicCheck', 'NoTechSupport', 'NoOnlineSecurity', 'ServiceRiskIndicator', 'HighRiskCustomer']


In [121]:
print(df["Churn"].value_counts())

Churn
0    5174
1    1869
Name: count, dtype: int64


In [122]:
print(df["Churn"].dtype)

int64


In [123]:
# Remove customer ID

X = df.drop(
    columns=["Churn", "customerID"]
)

y = df["Churn"]

print("X shape:", X.shape)
print("y shape:", y.shape)

X shape: (7043, 36)
y shape: (7043,)


In [124]:
y

0       0
1       0
2       1
3       0
4       1
       ..
7038    0
7039    0
7040    0
7041    1
7042    0
Name: Churn, Length: 7043, dtype: int64

In [125]:
# Identify feature types

numerical_columns = X.select_dtypes(
    include=["int64", "float64"]
).columns.tolist()

categorical_columns = X.select_dtypes(
    include=["object", "category"]
).columns.tolist()

print("Numerical features:")
print(numerical_columns)

print("\nCategorical features:")
print(categorical_columns)

Numerical features:
['SeniorCitizen', 'tenure', 'MonthlyCharges', 'TotalCharges', 'IsNewCustomer', 'IsLongTermCustomer', 'NumberOfServices', 'OptionalServiceCount', 'HasInternetService', 'IsFiberCustomer', 'ChargePerTenure', 'HighValueCustomer', 'MonthToMonthContract', 'ElectronicCheck', 'NoTechSupport', 'NoOnlineSecurity', 'ServiceRiskIndicator', 'HighRiskCustomer']

Categorical features:
['gender', 'Partner', 'Dependents', 'PhoneService', 'MultipleLines', 'InternetService', 'OnlineSecurity', 'OnlineBackup', 'DeviceProtection', 'TechSupport', 'StreamingTV', 'StreamingMovies', 'Contract', 'PaperlessBilling', 'PaymentMethod', 'TenureGroup', 'MonthlyChargeGroup', 'TotalChargeGroup']


/tmp/ipykernel_334668/1400397455.py:7: Pandas4Warning: For backward compatibility, 'str' dtypes are included by select_dtypes when 'object' dtype is specified. This behavior is deprecated and will be removed in a future version. Explicitly pass 'str' to `include` to select them, or to `exclude` to remove them and silence this warning.
See https://pandas.pydata.org/docs/user_guide/migration-3-strings.html#string-migration-select-dtypes for details on how to write code that works with pandas 2 and 3.
  categorical_columns = X.select_dtypes(


In [126]:
# Train-test split with stratification

X_train, X_test, y_train, y_test = train_test_split(
    X,
    y,
    test_size=0.20,
    random_state=42,
    stratify=y
)

print("X_train:", X_train.shape)
print("X_test :", X_test.shape)
print("y_train:", y_train.shape)
print("y_test :", y_test.shape)

X_train: (5634, 36)
X_test : (1409, 36)
y_train: (5634,)
y_test : (1409,)


In [127]:
X_train_encoded = pd.get_dummies(
    X_train,
    columns=categorical_columns,
    drop_first=True
)

X_test_encoded = pd.get_dummies(
    X_test,
    columns=categorical_columns,
    drop_first=True
)

In [128]:
X_train_encoded = pd.get_dummies(
    X_train,
    columns=categorical_columns,
    drop_first=True
)

X_test_encoded = pd.get_dummies(
    X_test,
    columns=categorical_columns,
    drop_first=True
)

In [129]:
scaler = StandardScaler()

X_train_encoded[numerical_columns] = scaler.fit_transform(
    X_train_encoded[numerical_columns]
)

X_test_encoded[numerical_columns] = scaler.transform(
    X_test_encoded[numerical_columns]
)

In [130]:
print("Final X_train shape:", X_train_encoded.shape)
print("Final X_test shape :", X_test_encoded.shape)

print(
    "Missing values in X_train:",
    X_train_encoded.isnull().sum().sum()
)

print(
    "Missing values in X_test:",
    X_test_encoded.isnull().sum().sum()
)

Final X_train shape: (5634, 53)
Final X_test shape : (1409, 53)
Missing values in X_train: 0
Missing values in X_test: 0


In [131]:
def calculate_metrics(
    model_name,
    y_true,
    y_pred,
    y_probability,
    training_time,
    prediction_time
):
    
    accuracy = accuracy_score(
        y_true,
        y_pred
    )

    precision = precision_score(
        y_true,
        y_pred,
        zero_division=0
    )

    recall = recall_score(
        y_true,
        y_pred,
        zero_division=0
    )

    f1 = f1_score(
        y_true,
        y_pred,
        zero_division=0
    )

    roc_auc = roc_auc_score(
        y_true,
        y_probability
    )

    pr_auc = average_precision_score(
        y_true,
        y_probability
    )

    print("\n==============================")
    print(model_name)
    print("==============================")

    print("Accuracy :", round(accuracy, 4))
    print("Precision:", round(precision, 4))
    print("Recall   :", round(recall, 4))
    print("F1 Score :", round(f1, 4))
    print("ROC-AUC  :", round(roc_auc, 4))
    print("PR-AUC   :", round(pr_auc, 4))

    print(
        "Training Time:",
        round(training_time, 4),
        "seconds"
    )

    print(
        "Prediction Time:",
        round(prediction_time, 4),
        "seconds"
    )

    return {
        "Model": model_name,
        "Accuracy": accuracy,
        "Precision": precision,
        "Recall": recall,
        "F1": f1,
        "ROC_AUC": roc_auc,
        "PR_AUC": pr_auc,
        "Training_Time": training_time,
        "Prediction_Time": prediction_time
    }

In [132]:
logistic_model = LogisticRegression(
    max_iter=1000,
    random_state=42
)

start_time = time.time()

logistic_model.fit(
    X_train_encoded,
    y_train
)

logistic_training_time = (
    time.time() - start_time
)

In [133]:
start_time = time.time()

logistic_pred = logistic_model.predict(
    X_test_encoded
)

logistic_probability = (
    logistic_model.predict_proba(
        X_test_encoded
    )[:, 1]
)

logistic_prediction_time = (
    time.time() - start_time
)

In [134]:
logistic_result = calculate_metrics(
    "Logistic Regression",
    y_test,
    logistic_pred,
    logistic_probability,
    logistic_training_time,
    logistic_prediction_time
)


Logistic Regression
Accuracy : 0.7991
Precision: 0.6564
Recall   : 0.5107
F1 Score : 0.5744
ROC-AUC  : 0.8434
PR-AUC   : 0.6397
Training Time: 0.5917 seconds
Prediction Time: 0.0364 seconds


In [135]:
logistic_cm = confusion_matrix(
    y_test,
    logistic_pred
)

print(logistic_cm)

[[935 100]
 [183 191]]


In [136]:
print(
    classification_report(
        y_test,
        logistic_pred,
        target_names=[
            "No Churn",
            "Churn"
        ]
    )
)

              precision    recall  f1-score   support

    No Churn       0.84      0.90      0.87      1035
       Churn       0.66      0.51      0.57       374

    accuracy                           0.80      1409
   macro avg       0.75      0.71      0.72      1409
weighted avg       0.79      0.80      0.79      1409



In [137]:
# Decision Tree

decision_tree_model = DecisionTreeClassifier(
    random_state=42
)

start_time = time.time()

decision_tree_model.fit(
    X_train_encoded,
    y_train
)

decision_tree_training_time = (
    time.time() - start_time
)

In [138]:
start_time = time.time()

decision_tree_pred = (
    decision_tree_model.predict(
        X_test_encoded
    )
)

decision_tree_probability = (
    decision_tree_model.predict_proba(
        X_test_encoded
    )[:, 1]
)

decision_tree_prediction_time = (
    time.time() - start_time
)

In [139]:
decision_tree_result = calculate_metrics(
    "Decision Tree",
    y_test,
    decision_tree_pred,
    decision_tree_probability,
    decision_tree_training_time,
    decision_tree_prediction_time
)


Decision Tree
Accuracy : 0.7367
Precision: 0.5039
Recall   : 0.5214
F1 Score : 0.5125
ROC-AUC  : 0.6677
PR-AUC   : 0.3898
Training Time: 0.1799 seconds
Prediction Time: 0.0184 seconds


In [140]:
# KNN

knn_model = KNeighborsClassifier(
    n_neighbors=5
)

start_time = time.time()

knn_model.fit(
    X_train_encoded,
    y_train
)

knn_training_time = (
    time.time() - start_time
)

In [141]:
start_time = time.time()

knn_pred = knn_model.predict(
    X_test_encoded
)

knn_probability = (
    knn_model.predict_proba(
        X_test_encoded
    )[:, 1]
)

knn_prediction_time = (
    time.time() - start_time
)

In [142]:
knn_result = calculate_metrics(
    "KNN",
    y_test,
    knn_pred,
    knn_probability,
    knn_training_time,
    knn_prediction_time
)


KNN
Accuracy : 0.7764
Precision: 0.5897
Recall   : 0.5187
F1 Score : 0.5519
ROC-AUC  : 0.7884
PR-AUC   : 0.5326
Training Time: 0.0127 seconds
Prediction Time: 0.3001 seconds


In [143]:
# Random Forest

random_forest_model = RandomForestClassifier(
    n_estimators=200,
    random_state=42,
    n_jobs=-1
)

start_time = time.time()

random_forest_model.fit(
    X_train_encoded,
    y_train
)

random_forest_training_time = (
    time.time() - start_time
)

In [144]:
start_time = time.time()

random_forest_pred = (
    random_forest_model.predict(
        X_test_encoded
    )
)

random_forest_probability = (
    random_forest_model.predict_proba(
        X_test_encoded
    )[:, 1]
)

random_forest_prediction_time = (
    time.time() - start_time
)

In [145]:
random_forest_result = calculate_metrics(
    "Random Forest",
    y_test,
    random_forest_pred,
    random_forest_probability,
    random_forest_training_time,
    random_forest_prediction_time
)


Random Forest
Accuracy : 0.7814
Precision: 0.6051
Recall   : 0.508
F1 Score : 0.5523
ROC-AUC  : 0.8223
PR-AUC   : 0.6116
Training Time: 1.976 seconds
Prediction Time: 0.3058 seconds


In [146]:
# Gradient Boosting

gradient_boosting_model = GradientBoostingClassifier(
    random_state=42
)

start_time = time.time()

gradient_boosting_model.fit(
    X_train_encoded,
    y_train
)

gradient_boosting_training_time = (
    time.time() - start_time
)

In [147]:
start_time = time.time()

gradient_boosting_pred = (
    gradient_boosting_model.predict(
        X_test_encoded
    )
)

gradient_boosting_probability = (
    gradient_boosting_model.predict_proba(
        X_test_encoded
    )[:, 1]
)

gradient_boosting_prediction_time = (
    time.time() - start_time
)

In [148]:
gradient_boosting_result = calculate_metrics(
    "Gradient Boosting",
    y_test,
    gradient_boosting_pred,
    gradient_boosting_probability,
    gradient_boosting_training_time,
    gradient_boosting_prediction_time
)


Gradient Boosting
Accuracy : 0.8013
Precision: 0.6667
Recall   : 0.5027
F1 Score : 0.5732
ROC-AUC  : 0.8425
PR-AUC   : 0.6514
Training Time: 2.5036 seconds
Prediction Time: 0.0416 seconds


In [149]:
# XGBoost

xgboost_model = XGBClassifier(
    n_estimators=200,
    max_depth=4,
    learning_rate=0.05,
    subsample=0.8,
    colsample_bytree=0.8,
    eval_metric="logloss",
    random_state=42,
    n_jobs=-1
)

start_time = time.time()

xgboost_model.fit(
    X_train_encoded,
    y_train
)

xgboost_training_time = (
    time.time() - start_time
)

In [150]:
start_time = time.time()

xgboost_pred = xgboost_model.predict(
    X_test_encoded
)

xgboost_probability = (
    xgboost_model.predict_proba(
        X_test_encoded
    )[:, 1]
)

xgboost_prediction_time = (
    time.time() - start_time
)

In [151]:
xgboost_result = calculate_metrics(
    "XGBoost",
    y_test,
    xgboost_pred,
    xgboost_probability,
    xgboost_training_time,
    xgboost_prediction_time
)


XGBoost
Accuracy : 0.7991
Precision: 0.6532
Recall   : 0.5187
F1 Score : 0.5782
ROC-AUC  : 0.8441
PR-AUC   : 0.6503
Training Time: 0.6977 seconds
Prediction Time: 0.0772 seconds


In [152]:
results = [
    logistic_result,
    decision_tree_result,
    knn_result,
    random_forest_result,
    gradient_boosting_result,
    xgboost_result
]

results_df = pd.DataFrame(results)

results_df

,Model,Accuracy,Precision,Recall,F1,ROC_AUC,PR_AUC,Training_Time,Prediction_Time
0,Logistic Regression,0.799148,0.656357,0.510695,0.574436,0.843362,0.639736,0.591665,0.036410
1,Decision Tree,0.736693,0.503876,0.521390,0.512484,0.667710,0.389757,0.179898,0.018361
2,KNN,0.776437,0.589666,0.518717,0.551920,0.788355,0.532609,0.012687,0.300129
3,Random Forest,0.781405,0.605096,0.508021,0.552326,0.822318,0.611618,1.976031,0.305828
4,Gradient Boosting,0.801278,0.666667,0.502674,0.573171,0.842478,0.651429,2.503611,0.041640
5,XGBoost,0.799148,0.653199,0.518717,0.578241,0.844127,0.650283,0.697680,0.077227


In [153]:
results_df = results_df.sort_values(
    by="ROC_AUC",
    ascending=False
)

results_df

,Model,Accuracy,Precision,Recall,F1,ROC_AUC,PR_AUC,Training_Time,Prediction_Time
5,XGBoost,0.799148,0.653199,0.518717,0.578241,0.844127,0.650283,0.697680,0.077227
0,Logistic Regression,0.799148,0.656357,0.510695,0.574436,0.843362,0.639736,0.591665,0.036410
4,Gradient Boosting,0.801278,0.666667,0.502674,0.573171,0.842478,0.651429,2.503611,0.041640
3,Random Forest,0.781405,0.605096,0.508021,0.552326,0.822318,0.611618,1.976031,0.305828
2,KNN,0.776437,0.589666,0.518717,0.551920,0.788355,0.532609,0.012687,0.300129
1,Decision Tree,0.736693,0.503876,0.521390,0.512484,0.667710,0.389757,0.179898,0.018361


In [154]:
results_df = results_df.sort_values(
    by="F1",
    ascending=False
)

results_df

,Model,Accuracy,Precision,Recall,F1,ROC_AUC,PR_AUC,Training_Time,Prediction_Time
5,XGBoost,0.799148,0.653199,0.518717,0.578241,0.844127,0.650283,0.697680,0.077227
0,Logistic Regression,0.799148,0.656357,0.510695,0.574436,0.843362,0.639736,0.591665,0.036410
4,Gradient Boosting,0.801278,0.666667,0.502674,0.573171,0.842478,0.651429,2.503611,0.041640
3,Random Forest,0.781405,0.605096,0.508021,0.552326,0.822318,0.611618,1.976031,0.305828
2,KNN,0.776437,0.589666,0.518717,0.551920,0.788355,0.532609,0.012687,0.300129
1,Decision Tree,0.736693,0.503876,0.521390,0.512484,0.667710,0.389757,0.179898,0.018361


In [155]:
results_df = results_df.sort_values(
    by="Recall",
    ascending=False
)

results_df

,Model,Accuracy,Precision,Recall,F1,ROC_AUC,PR_AUC,Training_Time,Prediction_Time
1,Decision Tree,0.736693,0.503876,0.521390,0.512484,0.667710,0.389757,0.179898,0.018361
5,XGBoost,0.799148,0.653199,0.518717,0.578241,0.844127,0.650283,0.697680,0.077227
2,KNN,0.776437,0.589666,0.518717,0.551920,0.788355,0.532609,0.012687,0.300129
0,Logistic Regression,0.799148,0.656357,0.510695,0.574436,0.843362,0.639736,0.591665,0.036410
3,Random Forest,0.781405,0.605096,0.508021,0.552326,0.822318,0.611618,1.976031,0.305828
4,Gradient Boosting,0.801278,0.666667,0.502674,0.573171,0.842478,0.651429,2.503611,0.041640


In [156]:
RESULT_PATH = (
    BASE_DIR
    / "data"
    / "processed"
    / "ml_model_comparison.csv"
)

results_df.to_csv(
    RESULT_PATH,
    index=False
)

print(
    "Model comparison saved to:"
)

print(RESULT_PATH)

Model comparison saved to:
/home/aximsoft/Downloads/Weekend_Task/AI Customer Intelligence Platform /data/processed/ml_model_comparison.csv


In [167]:
import joblib

best_models = {
    "Logistic Regression": logistic_model,
    "Decision Tree": decision_tree_model,
    "KNN": knn_model,
    "Random Forest": random_forest_model,
    "Gradient Boosting": gradient_boosting_model,
    "XGBoost": xgboost_model
}

best_model = best_models[
    "Gradient Boosting"
]

best_model_path = (
    BASE_DIR
    / "models"
    / "Gradient_Boosting.pkl"
)

joblib.dump(
    best_model,
    best_model_path
)

print(
    "Best model saved to:"
)

print(best_model_path)

Best model saved to:
/home/aximsoft/Downloads/Weekend_Task/AI Customer Intelligence Platform /models/Gradient_Boosting.pkl


In [160]:
PROCESSED_DIR = (
    BASE_DIR
    / "data"
    / "processed"
)

X_train_encoded.to_csv(
    PROCESSED_DIR / "ml_X_train_encoded.csv",
    index=False
)

X_test_encoded.to_csv(
    PROCESSED_DIR / "ml_X_test_encoded.csv",
    index=False
)

y_train.to_csv(
    PROCESSED_DIR / "ml_y_train.csv",
    index=False
)

y_test.to_csv(
    PROCESSED_DIR / "ml_y_test.csv",
    index=False
)

print("ML datasets saved.")

ML datasets saved.


In [161]:
X_train_encoded.to_csv(
    PROCESSED_DIR / "ml_X_train_final.csv",
    index=False
)

X_test_encoded.to_csv(
    PROCESSED_DIR / "ml_X_test_final.csv",
    index=False
)

print("Final ML datasets saved.")

Final ML datasets saved.
